In [ ]:
# Clone the repository
!git clone https://github.com/huggingface/lerobot.git

# Move into the folder
%cd lerobot

# Install LeRobot with SmolVLA dependencies
!pip install -e ".[smolvla]"

# Install server tools AND nest-asyncio for Colab compatibility
!pip install fastapi uvicorn pyngrok pydantic nest-asyncio

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Authenticate your ngrok tunnel (Requires adding 'ngrok_auth_token' to Colab Secrets)
ngrok.set_auth_token(userdata.get('ngrok_auth_token'))

# Open a tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print(f"--> SUCCESS! NGROK PUBLIC URL: {public_url}")

In [ ]:
import sys
import os
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch
import base64
import numpy as np
from io import BytesIO
from PIL import Image
import threading
import asyncio

# Apply nest_asyncio to allow nested event loops (for Colab)
nest_asyncio.apply()

sys.path.insert(0, "/content/lerobot/src")
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

app = FastAPI()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Server] Loading SmolVLA on {device}...")
sys.stdout.flush()

# Load your fine-tuned model
model_path = "leumals/smolvla_finetuned_v2"
policy = SmolVLAPolicy.from_pretrained(model_path, torch_dtype=torch.float32)

# Configure policy
policy.config.num_steps = 30
policy.config.chunk_size = 50
policy.config.n_action_steps = 50
policy.use_cache = False
policy.to(device)
policy.eval()

class InferenceRequest(BaseModel):
    image_agent: str
    image_wrist: str
    state: list[float]
    instruction: str

@app.post("/predict")
def predict(request: InferenceRequest):
    try:
        # 1. DECODE AS RGB AND SWAP TO BGR (Matches your OpenCV training data!)
        img_agent_pil = Image.open(BytesIO(base64.b64decode(request.image_agent))).convert("RGB")
        img_agent_bgr = np.array(img_agent_pil)[:, :, ::-1].copy()
        img_agent_tensor = torch.from_numpy(img_agent_bgr).permute(2,0,1).float() / 255.0

        img_wrist_pil = Image.open(BytesIO(base64.b64decode(request.image_wrist))).convert("RGB")
        img_wrist_bgr = np.array(img_wrist_pil)[:, :, ::-1].copy()
        img_wrist_tensor = torch.from_numpy(img_wrist_bgr).permute(2,0,1).float() / 255.0

        state_tensor = torch.tensor([request.state], dtype=torch.float32).to(device)

        # 2. Tokenize the text using the exact tokenizer from training
        tokenized_text = policy.model.vlm_with_expert.processor.tokenizer(
            request.instruction, return_tensors="pt", padding=True
        )

        # 3. Build the dictionary
        observation = {
            "observation.images.image": img_agent_tensor.unsqueeze(0).to(device),
            "observation.images.image2": img_wrist_tensor.unsqueeze(0).to(device),
            "observation.state": state_tensor,
            "observation.language.tokens": tokenized_text.input_ids.to(device),
            "observation.language.attention_mask": tokenized_text.attention_mask.to(device).bool()
        }

        # 4. Run inference
        with torch.no_grad():
            action_chunk = policy.select_action(observation)
            actions = action_chunk.cpu().numpy().tolist()
            return {"status": "success", "actions": actions}

    except Exception as e:
        import traceback
        traceback.print_exc()
        return {"status": "error", "message": str(e)}

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

print("Starting FastAPI server in background thread...")
threading.Thread(target=run_server, daemon=True).start()
print("Server is live 🎉 You can now send requests from Webots.")